# NB11 — Residual Training on Scaffold Split

This notebook trains the **canonical control-anchored residual-only model** on the
drug-level scaffold split created in NB10. It is the direct successor to NB01,
updated for the new split.

## What changes from NB01

| Aspect | NB01 | NB11 |
|--------|------|------|
| Split column | `split_ho_pathway` | `split_scaffold` (from NB10) |
| Split labels | `train/test/ood` mapped via `map_split()` | `train/test/ood/control` directly |
| PCA source | Fit on train cells inline | Re-fit from scratch here (train + controls only) |
| Main model | residual-only (canonical) | residual-only (canonical, unchanged) |
| MMD | Trained in parallel | Optional hook only, not foregrounded |

## Control policy (explicit)

Controls carry the sentinel label `split_scaffold = 'control'` from NB10.
In NB11 they are used as follows:

- **PCA fit**: train perturbed cells + all control cells (the 'control' sentinel is
  included in the fit mask). This matches NB10's gate convention.
- **Control means**: computed from all control cells, per cell type.
  Because controls are not drug-specific, they are available as anchors for
  every split without leakage.
- **Dataset class**: control cells are excluded from train/test/OOD batches —
  only perturbed cells are in the supervised residual learning sets.

## What gets saved

- `training_history_residual_only.csv`
- `final_overall_residual_only.csv`
- `final_per_cell_residual_only.csv`
- `best_residual_only.pt` (checkpoint)
- `pca_model_nb11.pkl` (train-only PCA, distinct from NB10 gate PCA)
- `drug_encoder_nb11.pkl`, `cell_encoder_nb11.pkl`
- `ctrl_means_pca_nb11.pkl`, `ctrl_means_gene_nb11.pkl`
- `nb11_run_summary.json`
- Optional: `final_overall_residual_mmd.csv`, `final_per_cell_residual_mmd.csv`

## 0) Drive + installs

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip -q install anndata scanpy torch pandas scikit-learn scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 152.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 163.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.


## 1) Config — all paths and hyperparameters in one place

In [3]:
import os
import json
import random
import pickle
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import anndata as ad
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score
from scipy.stats import pearsonr

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


@dataclass
class CFG:
    # ── paths ──────────────────────────────────────────────────────────────
    data_path: str     = "/content/drive/MyDrive/ChemDFM/data/sciplex_complete_middle_subset.h5ad"
    nb10_art_dir: str  = "/content/drive/MyDrive/ChemDFM/results_nb10_scaffold/artifacts"
    nb10_res_dir: str  = "/content/drive/MyDrive/ChemDFM/results_nb10_scaffold"
    results_dir: str   = "/content/drive/MyDrive/ChemDFM/results_nb11_scaffold"
    ckpt_dir: str      = "/content/drive/MyDrive/ChemDFM/checkpoints_nb11"

    # ── column names ───────────────────────────────────────────────────────
    condition_col: str = "condition"
    cell_col: str      = "cell_type"
    dose_col: str      = "dose"
    fallback_dose_col: str = "dose_val"
    split_col: str     = "split_scaffold"   # new column from NB10
    control_label: str = "control"

    # ── model / training ───────────────────────────────────────────────────
    pca_dim: int       = 25
    batch_size: int    = 512
    epochs: int        = 12
    lr: float          = 1e-3
    wd: float          = 1e-5
    emb_dim: int       = 32
    hidden: int        = 256
    dose_hidden: int   = 32
    alpha_topk: float  = 2.0
    alpha_mmd: float   = 0.05    # used only if MMD variant is run
    dropout: float     = 0.1
    pin_memory: bool   = False
    num_workers: int   = 0


cfg = CFG()
os.makedirs(cfg.results_dir, exist_ok=True)
os.makedirs(cfg.ckpt_dir, exist_ok=True)

print("Config:")
for k, v in asdict(cfg).items():
    print(f"  {k}: {v}")

Device: cuda
Config:
  data_path: /content/drive/MyDrive/ChemDFM/data/sciplex_complete_middle_subset.h5ad
  nb10_art_dir: /content/drive/MyDrive/ChemDFM/results_nb10_scaffold/artifacts
  nb10_res_dir: /content/drive/MyDrive/ChemDFM/results_nb10_scaffold
  results_dir: /content/drive/MyDrive/ChemDFM/results_nb11_scaffold
  ckpt_dir: /content/drive/MyDrive/ChemDFM/checkpoints_nb11
  condition_col: condition
  cell_col: cell_type
  dose_col: dose
  fallback_dose_col: dose_val
  split_col: split_scaffold
  control_label: control
  pca_dim: 25
  batch_size: 512
  epochs: 12
  lr: 0.001
  wd: 1e-05
  emb_dim: 32
  hidden: 256
  dose_hidden: 32
  alpha_topk: 2.0
  alpha_mmd: 0.05
  dropout: 0.1
  pin_memory: False
  num_workers: 0


## 2) Check NB10 gate and load scaffold split

NB11 loads `obs_with_scaffold_split.csv` from NB10's artifact directory.
This gives us the `split_scaffold` assignments without re-running scaffold logic.
We also check the NB10 gate JSON to confirm hard gates passed.

In [4]:
gate_path = os.path.join(cfg.nb10_res_dir, "scaffold_split_gate.json")
if os.path.exists(gate_path):
    with open(gate_path) as f:
        nb10_gate = json.load(f)
    print("NB10 gate results:")
    for k, v in nb10_gate.items():
        print(f"  {k}: {v}")
    if not nb10_gate.get("hard_gates_ok", False):
        raise RuntimeError(
            "NB10 hard gates did not pass. "
            "Fix NB10 before running NB11."
        )
    print("\nNB10 hard gates OK. Proceeding. ✓")
else:
    print(
        f"WARNING: NB10 gate file not found at {gate_path}.\n"
        "Proceeding without gate check — ensure NB10 ran successfully."
    )

obs_csv_path = os.path.join(cfg.nb10_art_dir, "obs_with_scaffold_split.csv")
if not os.path.exists(obs_csv_path):
    raise FileNotFoundError(
        f"FATAL: NB10 obs table not found at:\n  {obs_csv_path}\n"
        "Run NB10 first and ensure artifacts were saved."
    )
nb10_obs = pd.read_csv(obs_csv_path, index_col=0)
print(f"\nLoaded NB10 obs table: {nb10_obs.shape}")
print(f"Columns: {list(nb10_obs.columns[:15])} ...")
print(f"\nSplit counts from NB10:")
print(nb10_obs[cfg.split_col].value_counts())

NB10 gate results:
  A_schema: True
  B_splits_nonempty: True
  C_no_leakage: True
  D_baseline_positive: True
  E_bemis_murcko_available: True
  hard_gates_ok: True
  n_bemis: 187
  n_all_drugs: 187

NB10 hard gates OK. Proceeding. ✓

Loaded NB10 obs table: (354640, 36)
Columns: ['cell_type', 'dose', 'dose_character', 'dose_pattern', 'g1s_score', 'g2m_score', 'pathway', 'pathway_level_1', 'pathway_level_2', 'product_dose', 'product_name', 'proliferation_index', 'replicate', 'size_factor', 'target'] ...

Split counts from NB10:
split_scaffold
train      240527
test        51243
ood         49866
control     13004
Name: count, dtype: int64


## 3) Load AnnData and attach scaffold split

We load the raw `.h5ad` freshly and attach the `split_scaffold` column from NB10.
This avoids any dependency on NB10's in-memory state and ensures NB11 is
self-contained. PCA will be refit below from scratch on train-only cells.

In [5]:
adata = ad.read_h5ad(cfg.data_path)
print(adata)

# dose fallback
if cfg.dose_col not in adata.obs.columns:
    if cfg.fallback_dose_col in adata.obs.columns:
        print(f"INFO: using fallback dose column '{cfg.fallback_dose_col}'")
        adata.obs[cfg.dose_col] = adata.obs[cfg.fallback_dose_col]
    else:
        raise ValueError(
            f"FATAL: Neither '{cfg.dose_col}' nor '{cfg.fallback_dose_col}' "
            f"found. Available: {list(adata.obs.columns)}"
        )

# schema check
for col in [cfg.condition_col, cfg.cell_col, cfg.dose_col]:
    if col not in adata.obs.columns:
        raise ValueError(f"FATAL: Required column '{col}' missing from obs.")

# Attach scaffold split from NB10
# nb10_obs index should align with adata.obs index
if not nb10_obs.index.equals(adata.obs.index):
    # Try aligning by index intersection
    common = adata.obs.index.intersection(nb10_obs.index)
    if len(common) < len(adata.obs) * 0.95:
        raise ValueError(
            f"FATAL: NB10 obs index aligns poorly with adata.obs index. "
            f"Common: {len(common)}/{len(adata.obs)}. "
            "Ensure NB10 was run on the same h5ad file."
        )
    print(f"WARNING: Index partial mismatch; aligning on {len(common)} common rows.")
    adata = adata[common].copy()
    nb10_obs = nb10_obs.loc[common]

adata.obs[cfg.split_col] = nb10_obs[cfg.split_col].values

# Drop any rows with 'drop' label (drugs not in scaffold map)
n_before = len(adata)
adata = adata[adata.obs[cfg.split_col] != "drop"].copy()
n_after = len(adata)
if n_before != n_after:
    print(f"Dropped {n_before - n_after} cells labelled 'drop'.")

split_counts = adata.obs[cfg.split_col].value_counts()
print("\nFinal split counts:")
print(split_counts)

for req in ["train", "test", "ood", "control"]:
    assert req in split_counts.index and split_counts[req] > 0, \
        f"FATAL: Required split '{req}' is empty or missing."
print("\nAll required splits non-empty. ✓")

AnnData object with n_obs × n_vars = 354640 × 2000
    obs: 'cell_type', 'dose', 'dose_character', 'dose_pattern', 'g1s_score', 'g2m_score', 'pathway', 'pathway_level_1', 'pathway_level_2', 'product_dose', 'product_name', 'proliferation_index', 'replicate', 'size_factor', 'target', 'vehicle', 'batch', 'n_counts', 'dose_val', 'condition', 'drug_dose_name', 'cov_drug_dose_name', 'cov_drug', 'control', 'split_ho_pathway', 'split_tyrosine_ood', 'split_epigenetic_ood', 'split_cellcycle_ood', 'SMILES', 'split_ood_finetuning', 'split_ho_epigenetic', 'split_ho_epigenetic_all', 'split_random'
    var: 'id', 'num_cells_expressed-0-0', 'num_cells_expressed-1-0', 'num_cells_expressed-1', 'gene_id', 'in_lincs', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'all_DEGs', 'hvg', 'lincs_DEGs', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

Final split counts:
split_scaffold
train      240527
test        51243
ood  

## 4) Expression matrix + train-only PCA

**Leakage decision**: PCA is fit on `(split_scaffold == 'train') OR
(split_scaffold == 'control')` cells. This mirrors the NB10 gate convention
and the NB01 convention (where `train_mask` included both train perturbed and
control cells, since NB01 filtered controls before splitting).

Test and OOD cells are projected with `pca.transform()` only — no information
from them enters the PCA fit.

In [6]:
X = adata.X
if hasattr(X, "toarray"):
    X = X.toarray()
X = np.asarray(X, dtype=np.float32)

# PCA fit mask: train perturbed + all control cells
split_vals = adata.obs[cfg.split_col].values
train_pca_mask = np.isin(split_vals, ["train", "control"])
print(f"PCA fit: {train_pca_mask.sum():,} cells "
      f"(train perturbed + all controls — no test/OOD leakage)")

pca = PCA(n_components=cfg.pca_dim, random_state=SEED)
X_pca = np.zeros((adata.n_obs, cfg.pca_dim), dtype=np.float32)
X_pca[train_pca_mask]  = pca.fit_transform(X[train_pca_mask]).astype(np.float32)
X_pca[~train_pca_mask] = pca.transform(X[~train_pca_mask]).astype(np.float32)

print(f"PCA explained variance (top-{cfg.pca_dim}): {pca.explained_variance_ratio_.sum():.4f}")

# Save PCA — distinct from NB10 gate PCA
with open(f"{cfg.results_dir}/pca_model_nb11.pkl", "wb") as f:
    pickle.dump(pca, f)
np.save(f"{cfg.results_dir}/X_pca_nb11_{cfg.pca_dim}d.npy", X_pca)
print("PCA model saved.")

PCA fit: 253,531 cells (train perturbed + all controls — no test/OOD leakage)
PCA explained variance (top-25): 0.2873
PCA model saved.


## 5) Encoders, control means, and residuals

Control means are derived from all control cells (sentinel `split_scaffold == 'control'`).
Because controls are not drug-conditional, using all of them as anchors introduces no
leakage — a control cell's mean basal state tells us nothing about which drug is in test.

In [7]:
# Label encoders (fit on all cells so embedding table covers every drug/cell seen)
drug_enc = LabelEncoder()
cell_enc = LabelEncoder()
drug_enc.fit(adata.obs[cfg.condition_col].astype(str))
cell_enc.fit(adata.obs[cfg.cell_col].astype(str))

adata.obs["drug_idx"] = drug_enc.transform(adata.obs[cfg.condition_col].astype(str))
adata.obs["cell_idx"] = cell_enc.transform(adata.obs[cfg.cell_col].astype(str))

idx2cell = {i: c for i, c in enumerate(cell_enc.classes_)}
print("Cell types:", idx2cell)
print(f"n_drugs: {len(drug_enc.classes_)}  n_cells: {len(cell_enc.classes_)}")

# Per-cell-type control means from ALL control cells
control_mask = (adata.obs[cfg.split_col].values == "control")
ctrl_means = {}
ctrl_means_gene = {}
for cell in sorted(adata.obs[cfg.cell_col].astype(str).unique()):
    m = control_mask & (adata.obs[cfg.cell_col].astype(str).values == cell)
    if m.sum() == 0:
        raise ValueError(
            f"FATAL: No control cells for cell type '{cell}'. "
            "Cannot compute control anchor."
        )
    ctrl_means[cell] = X_pca[m].mean(axis=0).astype(np.float32)
    ctrl_means_gene[cell] = X[m].mean(axis=0).astype(np.float32)

print("\nControl cell counts by cell type:")
for cell in sorted(ctrl_means):
    n = (control_mask & (adata.obs[cfg.cell_col].astype(str).values == cell)).sum()
    print(f"  {cell}: {n:,}")

# Control-anchored residuals
X0_pca = np.stack(
    [ctrl_means[c] for c in adata.obs[cfg.cell_col].astype(str).values]
).astype(np.float32)
DELTA_pca = (X_pca - X0_pca).astype(np.float32)

# Save encoders and control means
with open(f"{cfg.results_dir}/drug_encoder_nb11.pkl", "wb") as f:
    pickle.dump(drug_enc, f)
with open(f"{cfg.results_dir}/cell_encoder_nb11.pkl", "wb") as f:
    pickle.dump(cell_enc, f)
with open(f"{cfg.results_dir}/ctrl_means_pca_nb11.pkl", "wb") as f:
    pickle.dump(ctrl_means, f)
with open(f"{cfg.results_dir}/ctrl_means_gene_nb11.pkl", "wb") as f:
    pickle.dump(ctrl_means_gene, f)
print("\nEncoders and control means saved.")

Cell types: {0: 'A549', 1: 'K562', 2: 'MCF7'}
n_drugs: 188  n_cells: 3

Control cell counts by cell type:
  A549: 3,287
  K562: 3,359
  MCF7: 6,358

Encoders and control means saved.


## 6) Dataset and DataLoaders

Identical structure to NB01's `ChemResidualDataset`, updated to use
`split_scaffold` instead of `_split3`. Controls are excluded from all
supervised batches.

In [8]:
class ChemResidualDataset(Dataset):
    """Perturbed cells only for a given scaffold split."""

    def __init__(self, split: str):
        mask = (
            (adata.obs[cfg.split_col].values == split)
            & (adata.obs[cfg.condition_col].astype(str).values != cfg.control_label)
        )
        self.idxs = np.where(mask)[0]
        if len(self.idxs) == 0:
            raise ValueError(f"FATAL: Dataset for split '{split}' is empty.")

    def __len__(self) -> int:
        return len(self.idxs)

    def __getitem__(self, i: int) -> dict:
        idx = self.idxs[i]
        row = adata.obs.iloc[idx]
        dose = np.log1p(max(float(row[cfg.dose_col]), 0.0))
        return {
            "x_true":   torch.tensor(X_pca[idx],    dtype=torch.float32),
            "x0":       torch.tensor(X0_pca[idx],   dtype=torch.float32),
            "delta":    torch.tensor(DELTA_pca[idx], dtype=torch.float32),
            "drug_idx": torch.tensor(int(row["drug_idx"]), dtype=torch.long),
            "cell_idx": torch.tensor(int(row["cell_idx"]), dtype=torch.long),
            "dose":     torch.tensor([dose], dtype=torch.float32),
            "condition": str(row[cfg.condition_col]),
            "cell_type": str(row[cfg.cell_col]),
        }


train_ds = ChemResidualDataset("train")
test_ds  = ChemResidualDataset("test")
ood_ds   = ChemResidualDataset("ood")

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
ood_loader   = DataLoader(ood_ds,   batch_size=cfg.batch_size, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)

print(f"Scaffold split dataset sizes (perturbed cells only):")
print(f"  train : {len(train_ds):,}")
print(f"  test  : {len(test_ds):,}")
print(f"  ood   : {len(ood_ds):,}")

Scaffold split dataset sizes (perturbed cells only):
  train : 240,527
  test  : 51,243
  ood   : 49,866


## 7) Model architecture

Identical to NB01 `ResidualDoseResponseModel`. No structural changes —
the scaffold split is a data change, not a model change.

In [9]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dims, out_dim, dropout=0.1):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class StructuredDoseEncoder(nn.Module):
    def __init__(self, out_dim=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, out_dim))

    def forward(self, dose):
        return self.net(dose)


class ResidualDoseResponseModel(nn.Module):
    """
    Control-anchored residual model.
    Input : (x0, drug_idx, cell_idx, dose)
    Output: (delta_hat, x_hat = x0 + delta_hat)
    """
    def __init__(self, latent_dim, n_drugs, n_cells,
                 emb_dim=32, hidden=256, dose_hidden=32, dropout=0.1):
        super().__init__()
        self.drug_emb  = nn.Embedding(n_drugs, emb_dim)
        self.cell_emb  = nn.Embedding(n_cells, emb_dim)
        self.dose_enc  = StructuredDoseEncoder(dose_hidden)
        self.ctrl_enc  = MLP(latent_dim, [hidden, hidden], hidden, dropout)
        fusion_in      = hidden + emb_dim + emb_dim + dose_hidden
        self.delta_head = MLP(fusion_in, [hidden, hidden], latent_dim, dropout)

    def forward(self, x0, drug_idx, cell_idx, dose):
        z = torch.cat([
            self.ctrl_enc(x0),
            self.drug_emb(drug_idx),
            self.cell_emb(cell_idx),
            self.dose_enc(dose),
        ], dim=1)
        delta_hat = self.delta_head(z)
        return delta_hat, x0 + delta_hat


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Verify architecture
_probe = ResidualDoseResponseModel(
    latent_dim=cfg.pca_dim,
    n_drugs=len(drug_enc.classes_),
    n_cells=len(cell_enc.classes_),
    emb_dim=cfg.emb_dim,
    hidden=cfg.hidden,
    dose_hidden=cfg.dose_hidden,
    dropout=cfg.dropout,
)
print(f"Model parameter count: {count_params(_probe):,}")
del _probe

Model parameter count: 307,513


## 8) Loss functions and metrics

Identical to NB01: MSE on residuals + top-k residual emphasis.
MMD functions are defined but only called when `alpha_mmd > 0`.

In [10]:
# ── Top-k mask (residual emphasis) ────────────────────────────────────────
def topk_mask_from_true(delta_true, k=10):
    idx = torch.topk(delta_true.abs(), k=min(k, delta_true.shape[1]), dim=1).indices
    mask = torch.zeros_like(delta_true)
    mask.scatter_(1, idx, 1.0)
    return mask


# ── Optional MMD (kept as a hook, not foregrounded) ────────────────────────
def _pairwise_sq_dists(x, y):
    xn = (x ** 2).sum(1, keepdim=True)
    yn = (y ** 2).sum(1, keepdim=True).T
    return torch.clamp(xn + yn - 2.0 * x @ y.T, min=0.0)


def rbf_mmd(x, y, gamma=None):
    if x.shape[0] < 2 or y.shape[0] < 2:
        return x.new_tensor(0.0)
    dxy = _pairwise_sq_dists(x, y)
    dxx = _pairwise_sq_dists(x, x)
    dyy = _pairwise_sq_dists(y, y)
    if gamma is None:
        with torch.no_grad():
            vals = dxy.flatten()
            vals = vals[vals > 0]
            gamma = 1.0 / (vals.median().item() + 1e-6) if vals.numel() > 0 else 1.0
    return (torch.exp(-gamma * dxx).mean()
            + torch.exp(-gamma * dyy).mean()
            - 2.0 * torch.exp(-gamma * dxy).mean())


def cell_aware_mmd(x_hat, x_true, cell_idx, min_count=8):
    losses = []
    for cid in torch.unique(cell_idx):
        m = (cell_idx == cid)
        if m.sum().item() >= min_count:
            losses.append(rbf_mmd(x_hat[m], x_true[m]))
    return torch.stack(losses).mean() if losses else x_hat.new_tensor(0.0)


# ── Evaluation metrics (identical to NB01) ─────────────────────────────────
def rowwise_pearson(a: np.ndarray, b: np.ndarray) -> float:
    vals = []
    for i in range(a.shape[0]):
        if np.std(a[i]) < 1e-8 or np.std(b[i]) < 1e-8:
            continue
        vals.append(pearsonr(a[i], b[i])[0])
    return float(np.mean(vals)) if vals else np.nan


def compute_metrics(pred: np.ndarray, true: np.ndarray, x0: np.ndarray) -> dict:
    out = {
        "r2_full":         float(r2_score(true.reshape(-1), pred.reshape(-1))),
        "pearson_rowmean": rowwise_pearson(true, pred),
        "mse":             float(np.mean((true - pred) ** 2)),
    }
    for k in [20, 50]:
        vals = []
        for i in range(true.shape[0]):
            idx = np.argsort(-np.abs(true[i] - x0[i]))[:k]
            vals.append(r2_score(true[i, idx], pred[i, idx]))
        out[f"r2_top{k}"] = float(np.mean(vals))
    return out


@torch.no_grad()
def evaluate_loader(model, loader, split_name: str):
    """Returns (overall_metrics_dict, per_cell_df)."""
    model.eval()
    all_pred, all_true, all_x0, all_cells = [], [], [], []
    for batch in loader:
        x0       = batch["x0"].to(DEVICE)
        x_true   = batch["x_true"].to(DEVICE)
        drug_idx = batch["drug_idx"].to(DEVICE)
        cell_idx = batch["cell_idx"].to(DEVICE)
        dose     = batch["dose"].to(DEVICE)
        _, x_hat = model(x0, drug_idx, cell_idx, dose)
        all_pred.append(x_hat.cpu().numpy())
        all_true.append(x_true.cpu().numpy())
        all_x0.append(x0.cpu().numpy())
        all_cells.extend(batch["cell_type"])

    pred  = np.concatenate(all_pred)
    true  = np.concatenate(all_true)
    x0_np = np.concatenate(all_x0)
    cells = np.array(all_cells)

    overall = compute_metrics(pred, true, x0_np)
    rows = []
    for cell in sorted(set(all_cells)):
        m = cells == cell
        rows.append({"split": split_name, "cell_type": cell,
                     **compute_metrics(pred[m], true[m], x0_np[m])})
    return overall, pd.DataFrame(rows)


print("Loss functions and metrics defined.")

Loss functions and metrics defined.


## 9) Training loop

Identical training loop to NB01's `train_one_model()`. Key points:

- Selection score weights test R²_top50 most heavily (0.50), with OOD and K562
  as secondary signals — same weighting as NB01.
- K562 tracking is kept even if K562 is not in every split (guarded with a try/except).
- Best checkpoint is saved whenever selection score improves.

In [11]:
def _k562_r2(per_cell_df: pd.DataFrame, col="r2_top50") -> float:
    """Safely extract K562 r2_top50; returns NaN if K562 not in split."""
    sub = per_cell_df[per_cell_df["cell_type"] == "K562"]
    if len(sub) == 0:
        return float("nan")
    return float(sub[col].iloc[0])


def train_one_model(alpha_mmd: float = 0.0, tag: str = "residual_only") -> dict:
    """
    Train a single model variant.
    alpha_mmd=0.0 → residual-only (canonical)
    alpha_mmd>0   → residual + cell-aware MMD
    Returns a summary dict of final metrics.
    """
    model = ResidualDoseResponseModel(
        latent_dim=cfg.pca_dim,
        n_drugs=len(drug_enc.classes_),
        n_cells=len(cell_enc.classes_),
        emb_dim=cfg.emb_dim,
        hidden=cfg.hidden,
        dose_hidden=cfg.dose_hidden,
        dropout=cfg.dropout,
    ).to(DEVICE)

    optimizer  = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
    best_score = -1e9
    hist       = []
    best_path  = os.path.join(cfg.ckpt_dir, f"best_{tag}.pt")

    print(f"\n{'='*60}")
    print(f"Training: {tag}  |  alpha_mmd={alpha_mmd}  |  epochs={cfg.epochs}")
    print(f"{'='*60}")

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        epoch_losses = []

        for batch in train_loader:
            x0         = batch["x0"].to(DEVICE)
            x_true     = batch["x_true"].to(DEVICE)
            delta_true = batch["delta"].to(DEVICE)
            drug_idx   = batch["drug_idx"].to(DEVICE)
            cell_idx   = batch["cell_idx"].to(DEVICE)
            dose       = batch["dose"].to(DEVICE)

            optimizer.zero_grad()
            delta_hat, x_hat = model(x0, drug_idx, cell_idx, dose)

            loss_mse  = F.mse_loss(delta_hat, delta_true)
            mask      = topk_mask_from_true(delta_true, k=10)
            loss_topk = (((delta_hat - delta_true) ** 2) * mask).sum() / (mask.sum() + 1e-8)
            loss_mmd  = (cell_aware_mmd(x_hat, x_true, cell_idx)
                         if alpha_mmd > 0 else x_hat.new_tensor(0.0))

            loss = loss_mse + cfg.alpha_topk * loss_topk + alpha_mmd * loss_mmd
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())

        # ── epoch evaluation ───────────────────────────────────────────
        test_metrics, test_per_cell = evaluate_loader(model, test_loader, "test")
        ood_metrics,  ood_per_cell  = evaluate_loader(model, ood_loader,  "ood")

        k562_test = _k562_r2(test_per_cell)
        k562_ood  = _k562_r2(ood_per_cell)

        # Selection score: same weighting as NB01
        score_parts = [
            0.50 * test_metrics["r2_top50"],
            0.25 * ood_metrics["r2_top50"],
        ]
        if not np.isnan(k562_test):
            score_parts.append(0.15 * k562_test)
        if not np.isnan(k562_ood):
            score_parts.append(0.10 * k562_ood)
        score = sum(score_parts)

        row = {
            "epoch":               epoch,
            "train_loss":          float(np.mean(epoch_losses)),
            "test_r2_top50":       test_metrics["r2_top50"],
            "test_r2_top20":       test_metrics["r2_top20"],
            "ood_r2_top50":        ood_metrics["r2_top50"],
            "ood_r2_top20":        ood_metrics["r2_top20"],
            "k562_test_r2_top50":  k562_test,
            "k562_ood_r2_top50":   k562_ood,
            "selection_score":     score,
        }
        hist.append(row)

        ckpt_marker = ""
        if score > best_score:
            best_score = score
            torch.save(model.state_dict(), best_path)
            ckpt_marker = "  ✅"

        print(
            f"Epoch {epoch:02d} | Loss {row['train_loss']:.4f} "
            f"| Test top50 {row['test_r2_top50']:.4f} "
            f"| OOD top50  {row['ood_r2_top50']:.4f} "
            f"| K562 test {row['k562_test_r2_top50']:.4f} "
            f"| K562 OOD  {row['k562_ood_r2_top50']:.4f}"
            + ckpt_marker
        )

    # ── final evaluation on best checkpoint ───────────────────────────
    hist_df = pd.DataFrame(hist)
    hist_df.to_csv(f"{cfg.results_dir}/training_history_{tag}.csv", index=False)

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    final_test, final_test_pc = evaluate_loader(model, test_loader, "test")
    final_ood,  final_ood_pc  = evaluate_loader(model, ood_loader,  "ood")

    final_overall = pd.DataFrame([
        {"split": "test", "model": tag, **final_test},
        {"split": "ood",  "model": tag, **final_ood},
    ])
    final_per_cell = pd.concat(
        [final_test_pc.assign(model=tag), final_ood_pc.assign(model=tag)],
        ignore_index=True
    )

    final_overall.to_csv(f"{cfg.results_dir}/final_overall_{tag}.csv", index=False)
    final_per_cell.to_csv(f"{cfg.results_dir}/final_per_cell_{tag}.csv", index=False)

    print(f"\nFinal overall metrics ({tag}):")
    print(final_overall[["split", "r2_top50", "r2_top20", "pearson_rowmean", "mse"]].to_string(index=False))
    print(f"\nFinal per-cell metrics ({tag}):")
    print(final_per_cell[["split", "cell_type", "r2_top50", "r2_top20"]].to_string(index=False))

    summary = {
        "tag":                    tag,
        "best_epoch":             int(hist_df.loc[hist_df["selection_score"].idxmax(), "epoch"]),
        "best_selection_score":   float(hist_df["selection_score"].max()),
        "final_test_r2_top50":    final_test["r2_top50"],
        "final_test_r2_top20":    final_test["r2_top20"],
        "final_ood_r2_top50":     final_ood["r2_top50"],
        "final_ood_r2_top20":     final_ood["r2_top20"],
        "final_k562_test_r2_top50": _k562_r2(final_test_pc),
        "final_k562_ood_r2_top50":  _k562_r2(final_ood_pc),
        "checkpoint":             best_path,
    }
    return summary


print("train_one_model() defined.")

train_one_model() defined.


## 10) Run residual-only (canonical)

This is the main run. Residual-only is the canonical baseline.

In [12]:
summary_ro = train_one_model(alpha_mmd=0.0, tag="residual_only")


Training: residual_only  |  alpha_mmd=0.0  |  epochs=12
Epoch 01 | Loss 1.8741 | Test top50 0.5165 | OOD top50  0.5286 | K562 test 0.5609 | K562 OOD  0.5652  ✅
Epoch 02 | Loss 1.8220 | Test top50 0.5099 | OOD top50  0.5229 | K562 test 0.5568 | K562 OOD  0.5607
Epoch 03 | Loss 1.8124 | Test top50 0.5040 | OOD top50  0.5179 | K562 test 0.5462 | K562 OOD  0.5520
Epoch 04 | Loss 1.8068 | Test top50 0.5078 | OOD top50  0.5259 | K562 test 0.5553 | K562 OOD  0.5637
Epoch 05 | Loss 1.8039 | Test top50 0.5071 | OOD top50  0.5202 | K562 test 0.5496 | K562 OOD  0.5534
Epoch 06 | Loss 1.8016 | Test top50 0.5150 | OOD top50  0.5315 | K562 test 0.5577 | K562 OOD  0.5661
Epoch 07 | Loss 1.7999 | Test top50 0.5028 | OOD top50  0.5188 | K562 test 0.5489 | K562 OOD  0.5528
Epoch 08 | Loss 1.7990 | Test top50 0.5112 | OOD top50  0.5293 | K562 test 0.5577 | K562 OOD  0.5658
Epoch 09 | Loss 1.7978 | Test top50 0.5201 | OOD top50  0.5320 | K562 test 0.5552 | K562 OOD  0.5609  ✅
Epoch 10 | Loss 1.7970 | Tes

## 11) Optional: residual + MMD

Run only if you want to replicate the NB01/NB03 comparison on the scaffold split.
Based on NB03 results, MMD gain was negligible; this run is not expected to
significantly beat residual-only and is not the canonical result.

In [13]:
# Uncomment to run. Adds ~30-40% training time on top of residual-only.
# summary_mmd = train_one_model(alpha_mmd=cfg.alpha_mmd, tag="residual_mmd")
summary_mmd = None
print("MMD run skipped (uncomment above to enable).")

MMD run skipped (uncomment above to enable).


## 12) Optional comparison: scaffold split vs old pathway split

If NB01 artifacts exist on disk, print a side-by-side comparison.
This block is fully optional and fails gracefully.

In [14]:
from pathlib import Path


def _find_latest(pattern: str, roots=("/content/drive/MyDrive/ChemDFM",)) -> str | None:
    found = []
    for root in roots:
        found.extend(Path(root).rglob(pattern) if Path(root).exists() else [])
    if not found:
        return None
    return str(sorted(found, key=lambda p: p.stat().st_mtime)[-1])


nb01_overall_path = _find_latest("final_overall_residual_only.csv")
nb01_cell_path    = _find_latest("final_per_cell_residual_only.csv")

# Exclude NB11's own freshly saved CSVs from the "old" comparison
def _exclude_nb11(path):
    if path and "results_nb11" in str(path):
        return None
    return path

nb01_overall_path = _exclude_nb11(nb01_overall_path)
nb01_cell_path    = _exclude_nb11(nb01_cell_path)

if nb01_overall_path and os.path.exists(nb01_overall_path):
    nb01_overall = pd.read_csv(nb01_overall_path)
    nb11_overall = pd.read_csv(f"{cfg.results_dir}/final_overall_residual_only.csv")

    print("=" * 70)
    print("COMPARISON: pathway split (NB01) vs scaffold split (NB11) — residual-only")
    print("=" * 70)

    for split in ["test", "ood"]:
        r_old = nb01_overall[nb01_overall["split"] == split]
        r_new = nb11_overall[nb11_overall["split"] == split]
        if len(r_old) == 0 or len(r_new) == 0:
            continue
        old_top50 = float(r_old["r2_top50"].iloc[0])
        new_top50 = float(r_new["r2_top50"].iloc[0])
        old_top20 = float(r_old["r2_top20"].iloc[0])
        new_top20 = float(r_new["r2_top20"].iloc[0])
        print(f"\n  [{split.upper()}]")
        print(f"    Pathway split (NB01) : r2_top50={old_top50:.4f}  r2_top20={old_top20:.4f}")
        print(f"    Scaffold split (NB11) : r2_top50={new_top50:.4f}  r2_top20={new_top20:.4f}")
        print(f"    Δ r2_top50            : {new_top50 - old_top50:+.4f}")

    if nb01_cell_path and os.path.exists(nb01_cell_path):
        nb01_cell = pd.read_csv(nb01_cell_path)
        nb11_cell = pd.read_csv(f"{cfg.results_dir}/final_per_cell_residual_only.csv")
        print("\n  Per-cell delta (scaffold - pathway) for r2_top50:")
        for split in ["test", "ood"]:
            old_c = nb01_cell[nb01_cell["split"] == split].set_index("cell_type")["r2_top50"]
            new_c = nb11_cell[nb11_cell["split"] == split].set_index("cell_type")["r2_top50"]
            common = old_c.index.intersection(new_c.index)
            delta = (new_c[common] - old_c[common]).rename("delta")
            print(f"  [{split}]:")
            for ct, d in delta.sort_values().items():
                print(f"    {ct:20s} {d:+.4f}")
else:
    print(
        "NB01 artifacts not found — skipping comparison block.\n"
        "This is expected if NB01 was not run or saved to a different path."
    )

NB01 artifacts not found — skipping comparison block.
This is expected if NB01 was not run or saved to a different path.


## 13) Save run summary JSON

In [15]:
run_summary = {
    "notebook":         "NB11_Residual_Training_on_ScaffoldSplit",
    "split_col":        cfg.split_col,
    "pca_dim":          cfg.pca_dim,
    "pca_source":       "train_perturbed_plus_controls__no_test_ood_leakage",
    "n_train":          len(train_ds),
    "n_test":           len(test_ds),
    "n_ood":            len(ood_ds),
    "n_drugs":          int(len(drug_enc.classes_)),
    "n_cells":          int(len(cell_enc.classes_)),
    "epochs":           cfg.epochs,
    "residual_only":    summary_ro,
    "residual_mmd":     summary_mmd,
    "artifacts": {
        "pca":          f"{cfg.results_dir}/pca_model_nb11.pkl",
        "drug_enc":     f"{cfg.results_dir}/drug_encoder_nb11.pkl",
        "cell_enc":     f"{cfg.results_dir}/cell_encoder_nb11.pkl",
        "ctrl_pca":     f"{cfg.results_dir}/ctrl_means_pca_nb11.pkl",
        "ctrl_gene":    f"{cfg.results_dir}/ctrl_means_gene_nb11.pkl",
        "checkpoint":   os.path.join(cfg.ckpt_dir, "best_residual_only.pt"),
        "history":      f"{cfg.results_dir}/training_history_residual_only.csv",
        "overall":      f"{cfg.results_dir}/final_overall_residual_only.csv",
        "per_cell":     f"{cfg.results_dir}/final_per_cell_residual_only.csv",
    },
}

summary_path = f"{cfg.results_dir}/nb11_run_summary.json"
with open(summary_path, "w") as f:
    json.dump(run_summary, f, indent=2, default=str)
print(f"Run summary saved to: {summary_path}")

Run summary saved to: /content/drive/MyDrive/ChemDFM/results_nb11_scaffold/nb11_run_summary.json


## Final report

In [16]:
print("=" * 60)
print("NB11 — RESIDUAL TRAINING ON SCAFFOLD SPLIT — SUMMARY")
print("=" * 60)
print(f"Split         : {cfg.split_col}")
print(f"PCA dim       : {cfg.pca_dim}")
print(f"PCA fit on    : train perturbed + all controls (no test/OOD)")
print(f"Train cells   : {len(train_ds):,}")
print(f"Test  cells   : {len(test_ds):,}")
print(f"OOD   cells   : {len(ood_ds):,}")
print()
print("=== Residual-Only (canonical) ===")
print(f"  Best epoch          : {summary_ro['best_epoch']}")
print(f"  Test  r2_top50      : {summary_ro['final_test_r2_top50']:.4f}")
print(f"  Test  r2_top20      : {summary_ro['final_test_r2_top20']:.4f}")
print(f"  OOD   r2_top50      : {summary_ro['final_ood_r2_top50']:.4f}")
print(f"  OOD   r2_top20      : {summary_ro['final_ood_r2_top20']:.4f}")
print(f"  K562  test r2_top50 : {summary_ro['final_k562_test_r2_top50']:.4f}")
print(f"  K562  OOD  r2_top50 : {summary_ro['final_k562_ood_r2_top50']:.4f}")
print()
print(f"Artifacts dir : {cfg.results_dir}")
print(f"Checkpoint    : {os.path.join(cfg.ckpt_dir, 'best_residual_only.pt')}")
print("\nNext: NB12_Mini_DrugConditioned_OT_Pilot.ipynb")

NB11 — RESIDUAL TRAINING ON SCAFFOLD SPLIT — SUMMARY
Split         : split_scaffold
PCA dim       : 25
PCA fit on    : train perturbed + all controls (no test/OOD)
Train cells   : 240,527
Test  cells   : 51,243
OOD   cells   : 49,866

=== Residual-Only (canonical) ===
  Best epoch          : 9
  Test  r2_top50      : 0.5201
  Test  r2_top20      : 0.4384
  OOD   r2_top50      : 0.5320
  OOD   r2_top20      : 0.4491
  K562  test r2_top50 : 0.5552
  K562  OOD  r2_top50 : 0.5609

Artifacts dir : /content/drive/MyDrive/ChemDFM/results_nb11_scaffold
Checkpoint    : /content/drive/MyDrive/ChemDFM/checkpoints_nb11/best_residual_only.pt

Next: NB12_Mini_DrugConditioned_OT_Pilot.ipynb
